# Global Developer Toolkit (`wt-nt8-eod`) – try it in your browser

Free, open-source (MIT) Python toolkit for **NinjaTrader 8 end-of-day text files** – including MetaStock®-format data converted with the free **WealthMetaStockToNT8 Converter**, from any market and any data vendor.
This notebook runs on the toolkit's **made-up sample data** (7 fictional symbols), so nothing needs to be downloaded or installed on your PC.

**Run:** *Runtime → Run all*, or click ▶ on each cell.

> ⚠ **Important:** This free tool is for education and analysis only – not investment advice or a recommendation to buy or sell any security. WealthTechnical LLP is not a SEBI-registered investment adviser or research analyst. The sample data is made up and is not real market data. Trading futures & options, crypto and other speculative instruments involves a high risk of loss. [Full disclaimer](https://www.wealthtechnical.com/disclaimer.html)

## 1 · Install the toolkit from PyPI

In [ ]:
!pip install -q wt-nt8-eod matplotlib

## 2 · Get the made-up sample data (0.5 MB, from GitHub)

In [ ]:
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/wealthtechnical-afk/metastock-to-ninjatrader8-converter.git wt_tk > /dev/null 2>&1
!cd wt_tk && git sparse-checkout set developer-toolkit/sample_data > /dev/null 2>&1
DATA = "wt_tk/developer-toolkit/sample_data"   # on your PC: the converter's output folder
print("sample data ready:", DATA)

## 3 · Load a whole converter output folder in one line

In [ ]:
import pandas as pd, wt_nt8_eod as wt
df = wt.read_output(DATA)            # reads NT-All/*.Last.txt  (yyyyMMdd;open;high;low;close;volume)
print(df.shape, "rows ·", df.symbol.nunique(), "symbols")
wt.summary(df)

## 4 · Wide table of closes and a quick chart

In [ ]:
closes = wt.to_wide(df, "close")
(closes / closes.iloc[0] * 100).plot(figsize=(10, 4), title="Rebased to 100 (made-up sample data)");

## 5 · Data-quality checks

In [ ]:
issues = wt.check_quality(df)       # duplicate dates, high < low, bad prices, long gaps, big jumps (possible unadjusted splits)
print(len(issues), "possible issues")
issues.head(15)

The sample data is clean, so here we break two bars on purpose to see the checks work:

In [ ]:
bad = df.copy()
bad.loc[bad.index[10], "high"] = bad.loc[bad.index[10], "low"] - 1     # high below low
bad = pd.concat([bad, bad.iloc[[20]]])                                  # duplicate date
wt.check_quality(bad)

## 6 · Symbol lists and mapping written by the converter

In [ ]:
print("import list:", wt.read_import_list(f"{DATA}/NT8-All Symbol-Import-List.txt")[:7])
mapping = wt.read_symbol_mapping(f"{DATA}/Symbol-Mapping.txt")
list(mapping.items())[:5]

## 7 · Export to other platforms

In [ ]:
import os
for fmt in ["amibroker", "metastock-csv", "mt5", "wealthtool"]:
    wt.WRITERS[fmt](df, f"out/{fmt}")
    print(f"{fmt:14s}", len(os.listdir(f"out/{fmt}")), "files")
wt.export(df, "out/all_bars.csv"); wt.to_sqlite(df, "out/eod.sqlite")
print("csv + sqlite written:", sorted(os.listdir("out")))

## Use it with your own data

1. Convert your MetaStock®-format EOD folders with the free **WealthMetaStockToNT8 Converter** (runs inside NinjaTrader 8 on Windows): https://www.wealthtechnical.com/wealthmetastock-to-nt8-converter.html – or use any NinjaTrader 8 EOD text files.
2. `pip install wt-nt8-eod`
3. `df = wt.read_output(r"C:\path\to\converter output")`

More: [GitHub & docs](https://github.com/wealthtechnical-afk/metastock-to-ninjatrader8-converter/tree/main/developer-toolkit) · [All free kits](https://www.wealthtechnical.com/free-data-kits/) · [India Data Kit for NSE/BSE (`pip install wt-india-eod`)](https://pypi.org/project/wt-india-eod/)

*For education and analysis only – not investment advice. MetaStock®, AmiBroker, MetaTrader and NinjaTrader® are trademarks of their owners. WealthTechnical LLP is an Ecosystem Vendor of NinjaTrader Group LLC; NinjaTrader does not endorse WealthTechnical products.*